# 04 — Лаборатория вертикальных спредов: четыре способа торговать направление

Вы построите все четыре вертикали, посчитаете их ограниченный риск, наложите **дебетовое и
кредитное** выражения *одного и того же* бычьего взгляда и проследите границу **POP против
максимальной прибыли** по коротким страйкам.

DEMO: спот **$100**, IV **0.25**, **45 DTE**. Середины рынка (mid) из цепочки модуля 00.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, viz


In [ ]:
SPOT, VOL, t = 100.0, 0.25, 45/365

def show(pos):
    s = analyzer.summarize(pos, SPOT, VOL)
    print(s['label'])
    print(f"  net_premium {s['net_premium']:+.0f}  breakevens {[round(b,2) for b in s['breakevens']]}")
    print(f"  max_profit {s['max_profit']:.0f}  max_loss {s['max_loss']:.0f}  POP {s['probability_of_profit']:.2f}")

## 1. Две дебетовые вертикали (бычья / медвежья, выражения для низкой IV)

In [ ]:
bull_call = strategies.bull_call_spread((100, 3.91), (110, 0.73), expiry=t)
bear_put  = strategies.bear_put_spread((100, 3.42), (90, 0.62), expiry=t)
show(bull_call); print(); show(bear_put)

Бычий колл-спред: рискуем **318** ради **682**, безубыточность 103.18. Максимальная прибыль +
максимальный убыток = ширина в $1000. Обе сделки дебетовые — вы заплатили за чистую опциональность
(небольшая длинная вега).

## 2. Две кредитные вертикали (бычья / медвежья, выражения для высокой IV)

In [ ]:
bull_put  = strategies.bull_put_spread((95, 1.58), (90, 0.62), expiry=t)
bear_call = strategies.bear_call_spread((105, 1.85), (110, 0.73), expiry=t)
show(bull_put); print(); show(bear_call)

Бычий пут-спред: собираем **96** и оставляем себе, если DEMO держится выше 95; рискуем **404**.
Обратите внимание на **более высокий POP**, чем у дебетовых спредов — вы зарабатываете, даже если
акция просто стоит на месте или немного сползает вниз.

## 3. Один взгляд, два выражения: дебет против кредита

Бычий колл-спред и бычий пут-спред **оба бычьи**. Наложите их выплаты на экспирации. Дебетовой
версии нужно движение вверх; кредитная выигрывает на «не вниз».

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_compare([bull_call, bull_put], ax=ax)
ax.set_title('Два бычьих выражения: бычий колл-спред (дебет) против бычьего пут-спреда (кредит)')
plt.show()

Выбор между ними — **решение по IV**: дебет (длинная вега) при низкой IV, кредит (короткая вега)
при высокой. Проверьте знаки веги по сводкам.

In [ ]:
gc = analyzer.summarize(bull_call, SPOT, VOL)['greeks']
gp = analyzer.summarize(bull_put,  SPOT, VOL)['greeks']
print(f'бычий КОЛЛ-спред  delta {gc.delta:+.1f}  vega {gc.vega:+.2f}  theta {gc.theta:+.2f}')
print(f'бычий ПУТ-спред   delta {gp.delta:+.1f}  vega {gp.vega:+.2f}  theta {gp.theta:+.2f}')

## 4. Правило управления «50% прибыли» (кредитный спред)

Кредитные спреды управляются на **50% от максимальной прибыли**. Через `payoff.pnl_at` посмотрите,
как выглядит переоценка P&L бычьего пут-спреда по модели по мере течения времени при неподвижной
акции — распад тянет позицию к цели в 50%.

In [ ]:
from optionslab import payoff
for days in [0, 10, 20, 30]:
    pnl = payoff.pnl_at(bull_put, SPOT, days/365, VOL)
    print(f'прошло дней: {days:2d}, спот стоит на 100:  P&L {pnl:+.0f}  (макс. прибыль 96)')

## 5. Граница «POP против максимальной прибыли»

Двигайте короткий страйк пута и наблюдайте размен: более близкие страйки платят больше, но
выигрывают реже. Максимизировать оба нельзя — выбирайте свою точку на границе.

In [ ]:
# (short_strike, short_prem, long_strike, long_prem) из цепочки DEMO на 45 DTE
variants = [(97.5, 2.37, 92.5, 1.01), (95, 1.58, 90, 0.62), (92.5, 1.01, 87.5, 0.37), (90, 0.62, 85, 0.22)]
for ks, ps, kl, pl in variants:
    sp = strategies.bull_put_spread((ks, ps), (kl, pl), expiry=t)
    s = analyzer.summarize(sp, SPOT, VOL)
    print(f'короткий {ks:5.1f}  кредит {-s["net_premium"]/100:4.2f}  POP {s["probability_of_profit"]:.2f}  max_profit {s["max_profit"]:.0f}  max_loss {s["max_loss"]:.0f}')

Читайте по столбцам: по мере ухода короткого страйка дальше OTM (с 97.5 к 90) **POP растёт**, а
**кредит / максимальная прибыль сокращаются**. Страйк около 30-дельты (примерно 95) — обычная точка
баланса.

## 6. Картина выплат кредитного спреда

`viz.plot_payoff` с текущими спотом и волатильностью размечает зону прибыли и точку безубыточности
бычьего пут-спреда.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
viz.plot_payoff(bull_put, spot=SPOT, vol=VOL, ax=ax)
ax.set_title('Бычий пут-спред — прибыль выше 94.04, ограниченный риск ниже')
plt.show()

## Эксперименты

1. В разделе 1 расширьте бычий колл-спред до **100/115** (короткий 115-й колл ~0.24). Как изменятся
   максимальная прибыль, максимальный убыток и безубыточность при большей ширине?
2. В разделе 3 добавьте в `plot_compare` одиночный **длинный 100-й колл**. Чем выплата голого колла
   отличается от двух спредов (неограниченная против ограниченной)?
3. В разделе 4 пересчитайте таблицу P&L по времени с акцией, сползающей **до 96**, вместо
   неподвижной. Хватает ли по-прежнему помощи от временного распада или дельта вредит сильнее?
4. В разделе 5 постройте аналогичную границу для **медвежьего колл-спреда** (двигайте короткий колл
   со 102.5 вверх до 110). Кредит на каждой дельте больше или меньше, чем на пут-стороне, и почему
   (скью)?
5. Пересчитайте сводку бычьего пут-спреда при `vol=0.40`. Насколько больше кредита вы бы собрали и
   почему это делает кредитные спреды стратегией для *высокой* IV?